In [1]:

# -*- coding: utf-8 -*-
import os
import re
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# CONFIGURACIÓN
# =========================
HEADLESS = True          # False si quieres ver el navegador
CHROMEDRIVER_PATH = None # Ej.: r"C:\ruta\al\chromedriver.exe" | None = Selenium Manager

# --- LÍDER ---
BASE_URL = "https://super.lider.cl"
PRODUCT_URL_TEMPLATE = "https://super.lider.cl/ip/cuidado-bucal/{sku}?from=/search"

# Pausas y timeouts (ajusta si el sitio está lento)
PAGE_TIMEOUT = 25
PAUSE_BETWEEN_SKUS = 1.0


# =========================
# HELPERS
# =========================
def clp_text_to_int(text):
    """Convierte texto tipo '$2.490' o '2490' a int 2490."""
    if not text:
        return None
    clean = re.sub(r"[^\d.,]", "", str(text))
    digits = re.sub(r"[.,]", "", clean)
    return int(digits) if digits.isdigit() else None

def format_clp(v):
    """Formatea int 2490 -> $2.490"""
    if v is None:
        return "None"
    return f"${v:,}".replace(",", ".")

def unique_preserve_order(seq):
    """Quita duplicados preservando orden."""
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_text(el):
    """Obtiene texto robusto desde un elemento (prioriza textContent)."""
    try:
        return (el.get_attribute("textContent") or el.text or "").strip()
    except:
        return ""

def try_click(driver, xpaths, timeout=3):
    """Intenta cerrar modales/cookies si aparecen (best effort)."""
    wait = WebDriverWait(driver, timeout)
    for xp in xpaths:
        try:
            btn = wait.until(EC.element_to_be_clickable((By.XPATH, xp)))
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.3)
            return True
        except:
            continue
    return False

def make_driver(headless=HEADLESS):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=es-CL")
    options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0 Safari/537.36")

    if CHROMEDRIVER_PATH:
        if not os.path.exists(CHROMEDRIVER_PATH):
            raise FileNotFoundError(f"No se encontró ChromeDriver en: {CHROMEDRIVER_PATH}")
        service = Service(executable_path=CHROMEDRIVER_PATH)
    else:
        service = Service()  # Selenium Manager

    driver = webdriver.Chrome(service=service, options=options)
    driver.implicitly_wait(2)
    return driver


# =========================
# LÍDER: ABRIR PDP POR SKU
# =========================
def open_product_page(driver, sku, timeout=PAGE_TIMEOUT):
    """
    Abre el PDP de Líder usando el patrón:
    https://super.lider.cl/ip/cuidado-bucal/{sku}?from=/search
    """
    url = PRODUCT_URL_TEMPLATE.format(sku=sku)
    driver.get(url)

    # Cierra posibles modales/cookies (best effort)
    try_click(driver, [
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'ACEPTAR')]",
        "//button[contains(., 'Acepto')]",
        "//button[contains(., 'Entendido')]",
        "//button[contains(., 'OK')]",
        "//*[@aria-label='cerrar' or @aria-label='close' or @aria-label='Close']",
        "//*[self::button][contains(@class,'close') or contains(@class,'Close')]",
    ], timeout=3)

    # Espera a que cargue contenedor de precio o un encabezado
    wait = WebDriverWait(driver, timeout)
    try:
        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "[data-testid='price-wrap'], [data-seo-id='hero-price']")
            )
        )
    except:
        # Igual devolvemos la URL para registro
        pass

    return url


# =========================
# LÍDER: PARSEO DEL PDP
# =========================
def parse_lider_pdp(driver):
    """
    Extrae:
    - Nombre Producto
    - URL Producto (current_url)
    - Precio Oferta (hero-price)
    - Precio Normal (strike-through-price)
    - Descuento % (calculado si hay oferta y normal)
    - Precios Detectados (lista: [oferta, normal] sin duplicados)
    - Precio Unitario Texto (hero-unit-price)
    """
    data = {
        "Nombre Producto": None,
        "URL Producto": driver.current_url,
        "Precio Oferta": None,
        "Precio Normal": None,
        "Descuento %": None,
        "Precios Detectados": None,
        "Precio Unitario Texto": None,
    }

    # Nombre de producto (varios posibles selectores robustos)
    name_selectors = [
        "h1[itemprop='name']",
        "h1[data-seo-id='hero-title']",
        "h1[data-seo-id='product-title']",
        "h1",
    ]
    for sel in name_selectors:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            name_txt = safe_text(el)
            if name_txt:
                data["Nombre Producto"] = name_txt
                break
        except:
            continue

    # Precio oferta (actual)
    offer_txt = None
    offer_selectors = [
        "[data-testid='price-wrap'] [data-seo-id='hero-price']",
        "[data-seo-id='hero-price']",
        "[data-fs-element='price'] [itemprop='price']",
        "[itemprop='price']",
    ]
    for sel in offer_selectors:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            offer_txt = safe_text(el)
            if offer_txt and "$" in offer_txt:
                break
        except:
            continue
    offer_val = clp_text_to_int(offer_txt)
    if isinstance(offer_val, int):
        data["Precio Oferta"] = offer_val

    # Precio normal (tachado)
    normal_txt = None
    try:
        el = driver.find_element(By.CSS_SELECTOR, "[data-seo-id='strike-through-price']")
        normal_txt = safe_text(el)
    except:
        normal_txt = None
    normal_val = clp_text_to_int(normal_txt)
    if isinstance(normal_val, int):
        data["Precio Normal"] = normal_val

    # Unit price (texto, p.ej. "$1.990 x un")
    unit_txt = None
    try:
        el = driver.find_element(By.CSS_SELECTOR, "[data-seo-id='hero-unit-price']")
        unit_txt = " ".join(safe_text(el).split())
    except:
        unit_txt = None
    if unit_txt:
        data["Precio Unitario Texto"] = unit_txt

    # Descuento calculado
    if data["Precio Oferta"] and data["Precio Normal"] and data["Precio Normal"] > 0:
        try:
            pct = round((1 - (data["Precio Oferta"] / data["Precio Normal"])) * 100)
            data["Descuento %"] = max(0, min(100, int(pct)))
        except:
            pass

    # Precios detectados (oferta y normal, sin duplicados)
    prices = []
    for v in (data["Precio Oferta"], data["Precio Normal"]):
        if isinstance(v, int):
            prices.append(v)
    data["Precios Detectados"] = unique_preserve_order(prices) if prices else None

    return data


# =========================
# SCRAPING POR SKU (LÍDER)
# =========================
def scrape_sku(driver, sku, timeout=PAGE_TIMEOUT):
    """
    Líder: 1 SKU -> 1 PDP -> 1 fila
    """
    used_url = open_product_page(driver, sku, timeout=timeout)

    # Si la página no cargó nada útil, devolver sin_resultados
    # (aun así, intentamos parsear algo)
    info = parse_lider_pdp(driver)

    estado = "ok" if any([
        info.get("Precio Oferta"),
        info.get("Precio Normal"),
        info.get("Precios Detectados"),
        info.get("Precio Unitario Texto"),
        info.get("Descuento %") is not None,
        info.get("Nombre Producto")
    ]) else "sin_resultados"

    row = {
        "SKU": sku,
        "URL Busqueda": used_url,   # aquí guardamos el PDP utilizado
        "Nombre Producto": info.get("Nombre Producto"),
        "URL Producto": info.get("URL Producto"),
        "Precio Oferta": info.get("Precio Oferta"),
        "Precio Normal": info.get("Precio Normal"),
        "Descuento %": info.get("Descuento %"),
        "Precio Unitario Texto": info.get("Precio Unitario Texto"),
        "Precios Detectados": info.get("Precios Detectados"),
        "Estado": estado
    }
    return [row]


def resumen_rows(rows):
    if not rows:
        return "sin_resultados", []
    estados = [r.get("Estado") for r in rows]
    estado = "ok" if any(e == "ok" for e in estados) else (estados[0] or "sin_precio")

    precios = []
    for r in rows:
        pdets = r.get("Precios Detectados")
        if isinstance(pdets, list):
            for v in pdets:
                if isinstance(v, int) and v not in precios:
                    precios.append(v)
        else:
            for k in ("Precio Oferta", "Precio Normal"):
                v = r.get(k)
                if isinstance(v, int) and v not in precios:
                    precios.append(v)
    return estado, precios


# =========================
# SCRAPING POR LISTA DE SKUs
# =========================
def scrape_skus(skus):
    driver = make_driver(headless=HEADLESS)
    all_rows = []
    total = len(skus)

    try:
        for i, sku in enumerate(skus, start=1):
            print(f"[{i}/{total}] Consultando SKU: {sku} ...")
            rows = scrape_sku(driver, sku, timeout=PAGE_TIMEOUT)

            estado, precios = resumen_rows(rows)
            precios_txt = ", ".join(format_clp(p) for p in precios) if precios else "None"
            print(f"    → Estado: {estado} | Precios detectados: {precios_txt}")

            all_rows.extend(rows)
            time.sleep(PAUSE_BETWEEN_SKUS)

    finally:
        driver.quit()

    df = pd.DataFrame(all_rows)

    # Orden de columnas
    cols = [
        "SKU", "Nombre Producto", "Precio Oferta", "Precio Normal",
        "Descuento %", "Precio Unitario Texto", "Precios Detectados",
        "URL Producto", "URL Busqueda", "Estado"
    ]
    for c in cols:
        if c not in df.columns:
            df[c] = None

    # Para Excel: pasar lista de precios a texto "2490;3150"
    df["Precios Detectados"] = df["Precios Detectados"].apply(
        lambda x: ";".join(str(v) for v in x) if isinstance(x, list) else None
    )

    return df[cols]


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # EJEMPLO: SKU de Líder (el que enviaste)
    
    skus_raw = [
        "00779464017132",
        "00779464017134",
        "00779464017133",
        "00789601551617",
        "00780490000063",
        "00780490000066",
        "00505456304143",
        "00779464017231",
        "00780490000098",
        "00780490000053",
        "00780490000061",
        "00780490000055",
        "00780490000033",
        "00780490092046",
        "00780490000062",
        "00780490000056",
        "00780490000059",
        "00779464017175",
        "00779464017273",
        "00779464017324",
        "00780490063250",
        "00780490063245",
        "00780490066753",
        "00780490063239",
        "00779464017262",
        "00779464063050",
        "00779464017261",
        "00779464063049",
        "00780490061095",
        "00780490000070",
        "00780490000004",
        "00780490000003",
        "00779464017158",
        "00779464017250",
        "00779464017237",
        "00779464017242",
        "00780490000038",
        "00780490000040",
        "00780490000092",
        "00789601552773",
        "00789600941929",
        "00789601552004",
        "00750106506458",
        "00779464017072",
        "00779464017247",
        "00779464017155",
        "00779464017013",
        "00780490000090",
        "00789600949809",
        "00779464017241",
        "00789600949858",
        "00789600949879",
    ]


    # Deduplicar preservando orden
    seen = set()
    skus = []
    for s in skus_raw:
        s2 = str(s).strip()
        if s2 and s2 not in seen:
            seen.add(s2)
            skus.append(s2)

    print(f"Total SKUs (sin duplicados): {len(skus)}")

    df = scrape_skus(skus)

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    output_file = f"precios_lider_{timestamp}.xlsx"
    df.to_excel(output_file, index=False)

    print(f"Archivo exportado en: {os.path.abspath(output_file)}")


Total SKUs (sin duplicados): 52


There was an error managing chromedriver (Unsuccessful response (403 Forbidden) for URL https://storage.googleapis.com/chrome-for-testing-public/146.0.7680.31/win64/chromedriver-win64.zip); using driver found in the cache


[1/52] Consultando SKU: 00779464017132 ...
    → Estado: ok | Precios detectados: $2.890, $3.990
[2/52] Consultando SKU: 00779464017134 ...
    → Estado: ok | Precios detectados: $2.890, $3.990
[3/52] Consultando SKU: 00779464017133 ...
    → Estado: ok | Precios detectados: $2.890, $3.990
[4/52] Consultando SKU: 00789601551617 ...
    → Estado: ok | Precios detectados: $1.990, $2.750
[5/52] Consultando SKU: 00780490000063 ...
    → Estado: ok | Precios detectados: $3.390
[6/52] Consultando SKU: 00780490000066 ...
    → Estado: ok | Precios detectados: $1.990, $3.390
[7/52] Consultando SKU: 00505456304143 ...
    → Estado: ok | Precios detectados: $1.990, $3.390
[8/52] Consultando SKU: 00779464017231 ...
    → Estado: ok | Precios detectados: None
[9/52] Consultando SKU: 00780490000098 ...
    → Estado: ok | Precios detectados: $3.890
[10/52] Consultando SKU: 00780490000053 ...
    → Estado: ok | Precios detectados: $1.000, $1.290
[11/52] Consultando SKU: 00780490000061 ...
    → Estad